In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../"))

In [ ]:
def read_csv_file(file_path):
    """
    Reads a CSV file from the given path and returns a pandas DataFrame.

    Parameters:
    file_path (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


run_1 = read_csv_file("../../data/evaluation/classifier-n_dist/run_1.csv")
run_2 = read_csv_file("../../data/evaluation/classifier-n_dist/run_2.csv")
run_3 = read_csv_file("../../data/evaluation/classifier-n_dist/run_3.csv")

baseline_run_1 = read_csv_file("../../data/evaluation/classifier-basic/run_1.csv")
baseline_run_2 = read_csv_file("../../data/evaluation/classifier-basic/run_2.csv")
baseline_run_3 = read_csv_file("../../data/evaluation/classifier-basic/run_3.csv")

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
n_dist = ["0", "1"]

architectures = [
    r"$\mathrm{Cla}_1$",
    r"$\mathrm{Cla}_2$",
    r"$\mathrm{Cla}_3$",
    r"$\mathrm{Cla}_4$",
]

classes = ["Ball", "Elfmeterpunkt", "Linienkreuzung"]

In [ ]:
n_dist = [0, 1]
runs = [run_1, run_2, run_3]
architectures = ["conv_v1", "conv_v2", "conv_v3", "conv_v4"]

cla_runs_1 = {}

for i, arch in enumerate(architectures):
    for j, run in enumerate(runs):
        cla_runs_1[f"cla{i}_run{j}"] = run[run["architecture"] == arch]


baseline_runs = [baseline_run_1, baseline_run_2, baseline_run_3]
cla_runs_0 = {}
for i, arch in enumerate(architectures):
    for j, run in enumerate(baseline_runs):
        cla_runs_0[f"cla{i}_run{j}"] = run[run["architecture"] == arch]

In [ ]:
print(cla_runs_1.keys())

print(cla_runs_1[f"cla{2}_run{2}"])

In [ ]:
print(float(cla_runs_1[f"cla{2}_run{2}"]["ball_ap_pooled"].iloc[0]))

print(
    [
        float(cla_runs_1[f"cla{cla}_run{i}"]["ball_ap_pooled"].iloc[0])
        for cla in [2]
        for i in [2]
    ]
)

In [ ]:
ceilings = {
    "Ball": float(cla_runs_0["cla1_run1"]["encoder_recall_at_k_ball"].iloc[0]),
    "Elfmeterpunkt": float(cla_runs_0["cla1_run1"]["encoder_recall_at_k_penaltyMark"].iloc[0]),
    "Linienkreuzung": float(cla_runs_0["cla1_run1"]["encoder_recall_at_k_intersections"].iloc[0]),
}

ap_values = {
    "Ball": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["ball_ap_pooled"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["ball_ap_pooled"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
    "Elfmeterpunkt": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["penaltyMark_ap_pooled"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["penaltyMark_ap_pooled"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
    "Linienkreuzung": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["intersections_mAP"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["intersections_mAP"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
}

euc_values = {
    "Ball": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["classifier_euc_error_ball"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["classifier_euc_error_ball"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
    "Elfmeterpunkt": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["classifier_euc_error_penaltyMark"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["classifier_euc_error_penaltyMark"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
    "Linienkreuzung": [
        [
            float(cla_runs_0[f"cla{cla}_run{i}"]["classifier_euc_error_intersections"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
        [
            float(cla_runs_1[f"cla{cla}_run{i}"]["classifier_euc_error_intersections"].iloc[0])
            for cla in range(len(architectures))
            for i in range(len(runs))
        ],
    ],
}


In [ ]:
# ── Reshape flat run lists into per-arch mean structure ────────────────────────
# ap_values[cls][ctx_idx] = flat list of len(archs) * len(runs) values
# We want: ap_data[arch][cls][ctx_idx] = list of run values (length = n_runs)

n_runs = len(runs)
n_archs = len(architectures)
dists = ["0", "1"]
n_dist_vals = [0, 1]

ap_data = {}
euc_data = {}

for i, arch in enumerate(architectures):
    ap_data[arch] = {cls: [] for cls in classes}
    euc_data[arch] = {cls: [] for cls in classes}
    for cls in classes:
        for ctx_idx in range(len(n_dist_vals)):
            run_vals_ap = ap_values[cls][ctx_idx][i * n_runs : (i + 1) * n_runs]
            run_vals_euc = euc_values[cls][ctx_idx][i * n_runs : (i + 1) * n_runs]
            ap_data[arch][cls].append(run_vals_ap)  # ap_data[arch][cls][ctx_idx] = [r0, r1, r2]
            euc_data[arch][cls].append(run_vals_euc)

# ── Shared layout parameters ───────────────────────────────────────────────────
x = np.arange(len(dists))
n_classes = len(classes)
bar_width = 0.22
offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
colors = ["#0072B2", "#E69F00", "#009E73"]
mean_color = "#CC79A7"

arch_labels_display = [
    r"$\mathrm{Cla}_1$",
    r"$\mathrm{Cla}_2$",
    r"$\mathrm{Cla}_3$",
    r"$\mathrm{Cla}_4$",
]

os.makedirs("../../plots/n_ctx", exist_ok=True)


# ── Helpers ────────────────────────────────────────────────────────────────────
def style_ax(ax):
    ax.set_xticks(x)
    ax.set_xticklabels(dists, fontsize=11)
    ax.set_xlabel(r"$n_{\mathrm{dist}}$", fontsize=11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)


def draw_ceilings(ax, ceiling_dict):
    for cls, color in zip(classes, colors, strict=True):
        ax.axhline(
            y=ceiling_dict[cls],
            color=color,
            linewidth=1.2,
            linestyle="--",
            alpha=0.8,
            zorder=2,
        )


def draw_points_and_mean(ax, data_dict):
    """
    data_dict[cls][ctx_idx] = list of run values
    Plots jittered scatter + mean hline + connecting line between ctx means.
    """
    for cls, color, offset in zip(classes, colors, offsets, strict=True):
        means = []
        for i, samples in enumerate(data_dict[cls]):
            samples = np.array(samples)
            mean = np.mean(samples)
            means.append(mean)

            # Jittered scatter points
            jitter = np.linspace(-0.06, 0.06, len(samples))
            ax.scatter(
                x[i] + offset + jitter,
                samples,
                color=color,
                s=30,
                zorder=5,
                alpha=0.7,
                edgecolors="white",
                linewidths=0.4,
            )
            # Mean hline
            ax.hlines(
                mean,
                x[i] + offset - 0.09,
                x[i] + offset + 0.09,
                colors=color,
                linewidth=2.5,
                zorder=6,
            )

        # Connecting line between the two ctx means
        ax.plot(
            x + offset,
            means,
            color=color,
            linewidth=0.7,
            linestyle="-",
            zorder=4,
            alpha=0.9,
        )


def add_legend(ax, with_ceiling=False):
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=color,
            markersize=7,
            label=cls,
        )
        for cls, color in zip(classes, colors, strict=True)
    ]
    if with_ceiling:
        handles += [
            plt.Line2D(
                [0],
                [0],
                color=color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.8,
                label=r"Recall$@K_c$ für " + f"{cls}",
            )
            for cls, color in zip(classes, colors, strict=True)
        ]
    ax.legend(
        handles=handles,
        loc="lower right",
        framealpha=0.9,
        fontsize=9,
        ncols=2 if with_ceiling else 1,
    )


# ── One AP + one Euc plot per architecture ─────────────────────────────────────
y_max_euc = (
    max(
        v
        for arch in architectures
        for cls in classes
        for ctx_idx in range(len(n_dist_vals))
        for v in euc_data[arch][cls][ctx_idx]
    )
    * 1.2
)

for arch, arch_label in zip(architectures, arch_labels_display, strict=True):
    # — AP / mAP ———————————————————————————————————————————————————————————————
    fig, ax1 = plt.subplots(figsize=(7, 4))
    # ax1.text(
    #     0.85,
    #     0.9,
    #     arch_label,
    #     transform=ax1.transAxes,
    #     fontsize=40,
    #     color="grey",
    #     alpha=0.35,
    #     ha="center",
    #     va="center",
    #     zorder=0,
    # )
    draw_points_and_mean(ax1, ap_data[arch])
    draw_ceilings(ax1, ceilings)
    ax1.set_ylim(0.4, 1)
    style_ax(ax1)
    ax1.set_ylabel("AP / mAP", fontsize=11)
    add_legend(ax1, with_ceiling=True)
    plt.tight_layout()
    plt.savefig(f"../../plots/n_dist/{arch}_ap.pdf")

    # — Euclidean error ————————————————————————————————————————————————————————
    fig, ax2 = plt.subplots(figsize=(7, 4))
    # ax2.text(
    #     0.85,
    #     0.9,
    #     arch_label,
    #     transform=ax2.transAxes,
    #     fontsize=40,
    #     color="grey",
    #     alpha=0.35,
    #     ha="center",
    #     va="center",
    #     zorder=0,
    # )
    draw_points_and_mean(ax2, euc_data[arch])
    ax2.set_ylim(0, 3.5)
    style_ax(ax2)
    ax2.set_ylabel(r"$\bar{e}^\prime$ [px]", fontsize=11)
    ax2.spines[["top"]].set_visible(False)
    add_legend(ax2, with_ceiling=False)
    plt.tight_layout()
    plt.savefig(f"../../plots/n_dist/{arch}_euc.pdf")

plt.show()